<a href="https://colab.research.google.com/github/SSK-KGP/ReLeaf/blob/main/Main_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
BASE_PATH = '/content/drive/MyDrive/PlantVillage/'
os.makedirs(BASE_PATH, exist_ok=True)

Mounted at /content/drive


In [2]:
LABELS_FILE = BASE_PATH + "class_names.json"
TFLITE_QUANT = BASE_PATH + "plant_disease_model_quant.tflite"
IMG_HEIGHT = 224
IMG_WIDTH = 224

In [3]:
!pip install ai-edge-litert

In [4]:
import io, json
import numpy as np
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter
from PIL import Image
from fastapi import FastAPI, File, UploadFile

app = FastAPI(title = "ReLeaf Plant Disease API")

try:
  interpreter = Interpreter(model_path = TFLITE_QUANT)
  interpreter.allocate_tensors()

  input_details = interpreter.get_input_details()
  output_details = interpreter.get_output_details()
except Exception as e:
  print(f"Error in loading the model {e}")
  interpreter = None

with open(LABELS_FILE, "r") as f:
  class_names = json.load(f)

def pre_process_image(image_bytes):
  img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
  img = img.resize((IMG_HEIGHT, IMG_WIDTH))

  input_type = input_details[0]["dtype"]
  if input_type == np.float32:
    img_array = np.array(img, dtype = np.float32) / 255.0
  else:
    img_array = np.array(img, dtype = np.uint8)

  return np.expand_dims(img_array, axis = 0)

In [5]:
@app.post('/predict')
async def product(file : UploadFile = File(...)):
  if interpreter is None:
    raise HTTPException(status_code = 500, detail = "Model not loaded")
  contents = await file.read()
  input_data = pre_process_image(contents)
  input_type = input_details[0]["dtype"]
  interpreter.set_tensor(input_details[0]['index'], input_data)
  interpreter.invoke()
  output_data = interpreter.get_tensor(output_details[0]['index'])
  prediction_idx = np.argsort(output_data[0])[-5:][::-1]
  raw_confidence = output_data[0]
  predictions = {}
  for i in prediction_idx:
    predictions[class_names[i]] = float(raw_confidence[i])
  return predictions

In [6]:
!pip install fastapi uvicorn pyngrok nest_asyncio

In [7]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio # Explicitly import asyncio

nest_asyncio.apply()

ngrok.set_auth_token("3D5gM5fqsN8kA5JNIXREVO4cwJK_7P3YBTdZFEW6gmQZDNHM")

try:
  public_url = ngrok.connect(8000)
  print(f"live at {public_url.public_url}/docs")
except Exception as e:
  print(f"ngrok may already be running")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop = 'asyncio')

server = uvicorn.Server(config=config)

loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())
#asyncio.run(server.serve())

#uvicorn.run(app, host="0.0.0.0", port=8000)

INFO:     Started server process [14733]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


live at https://footsore-disobey-rearview.ngrok-free.dev/docs
INFO:     2401:4900:88b3:a1d7:6dd8:6ff5:574:5780:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2401:4900:88b3:a1d7:6dd8:6ff5:574:5780:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2401:4900:88b3:a1d7:6dd8:6ff5:574:5780:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2401:4900:88b3:a1d7:6dd8:6ff5:574:5780:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2401:4900:88b3:a1d7:6dd8:6ff5:574:5780:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2401:4900:88b3:a1d7:6dd8:6ff5:574:5780:0 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [14733]


KeyboardInterrupt: 

In [8]:
ngrok.kill()